# Yahoo A1 · **Attention Is All You Need (from scratch)** · Encoder–Decoder · Supervised (Binary)

- **Model**: Hand-rolled **Transformer (Vaswani et al., 2017)** — Multi-Head Attention, FFN, Add&Norm, sinusoidal PE  
- **Data**: `./yahoo/A1Benchmark/real_*.csv` (univariate), RobustScaler (train-only), sliding window `[t−L..t−1] → y_t`  
- **Head**: Decoder consumes a single **BOS** token and outputs one **logit** for `y_t`  
- **Metrics**: Precision / Recall / F1 / Accuracy; **threshold chosen on validation (best F1)** and applied to test

## 0) Requirements
```
pip install torch numpy pandas scikit-learn
```


In [15]:
# 1) Configuration
from dataclasses import dataclass

@dataclass
class Config:
    a1_dir: str = "./TSB-AD-U/WSD"
    seq_len: int = 100
    # AIAINY Transformer hyperparams
    d_model: int = 64
    n_heads: int = 4
    n_enc_layers: int = 2
    n_dec_layers: int = 2
    dim_ff: int = 128
    dropout: float = 0.1
    # Training
    lr: float = 1e-3
    batch: int = 256
    epochs: int = 1000
    patience: int = 15
    device: str = "cuda"  # "cuda" or "cpu"
    # RobustScaler params
    robust_q_low: float = 25.0
    robust_q_high: float = 75.0
    robust_with_centering: bool = True
    robust_with_scaling: bool = True

cfg = Config()
cfg


Config(a1_dir='./TSB-AD-U/WSD', seq_len=100, d_model=64, n_heads=4, n_enc_layers=2, n_dec_layers=2, dim_ff=128, dropout=0.1, lr=0.001, batch=256, epochs=1000, patience=15, device='cuda', robust_q_low=25.0, robust_q_high=75.0, robust_with_centering=True, robust_with_scaling=True)

In [16]:
# 2) Imports
import os, glob, json
import numpy as np
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.5.1+cu124
CUDA available: True


## 3) Data utilities (RobustScaler + Sliding Window)

In [17]:
def _find_cols(df: pd.DataFrame):
    ts_candidates = {"timestamp", "time", "ts", "date"}
    val_candidates = {"value", "values", "metric"}
    lab_candidates = {"is_anomaly", "anomaly", "label", "isoutlier"}
    ts_col = next((c for c in df.columns if str(c).lower() in ts_candidates), None)
    val_col = next((c for c in df.columns if str(c).lower() in val_candidates), None)
    lab_col = next((c for c in df.columns if str(c).lower() in lab_candidates), None)
    if val_col is None or lab_col is None:
        raise ValueError(f"Required columns not found in CSV. Columns: {df.columns.tolist()}")
    return ts_col, val_col, lab_col

def load_yahoo_a1(csv_path: str):
    df = pd.read_csv(csv_path)
    ts_col, val_col, lab_col = _find_cols(df)
    if ts_col is not None:
        df = df.sort_values(ts_col)
    x = df[val_col].to_numpy(dtype=np.float32)
    y = df[lab_col].astype(int).to_numpy()
    return x, y
def load_wsd(csv_path: str):
    df = pd.read_csv(csv_path)
    data_col = df.columns[0]
    label_col = df.columns[1]
    x = df[data_col].to_numpy(dtype=np.float32)
    y = df[label_col].astype(int).to_numpy()
    return x, y
def fit_robust_scaler(x_train: np.ndarray, cfg):
    rs = RobustScaler(
        with_centering=cfg.robust_with_centering,
        with_scaling=cfg.robust_with_scaling,
        quantile_range=(cfg.robust_q_low, cfg.robust_q_high)
    )
    rs.fit(x_train.reshape(-1, 1))
    return rs

def apply_scaler(x: np.ndarray, scaler: RobustScaler):
    return scaler.transform(x.reshape(-1, 1)).astype(np.float32).squeeze(-1)

def make_windows(x: np.ndarray, y: np.ndarray, L: int):
    Xs, Ys, idx = [], [], []
    for t in range(L, len(x)):
        Xs.append(x[t - L:t])
        Ys.append(y[t])
        idx.append(t)
    X = np.asarray(Xs, dtype=np.float32)[..., None]  # [N, L, 1]
    Y = np.asarray(Ys, dtype=np.int64)               # [N]
    return X, Y, np.asarray(idx)

class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X)       # [N, L, 1]
        self.Y = torch.from_numpy(Y)       # [N]
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

def compute_pos_weight(y_train: np.ndarray) -> float:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    if pos == 0:
        return 1.0
    return float(neg / pos)


## 4) AIAINY Transformer (from scratch)

In [18]:
def subsequent_mask(size: int):
    attn_shape = (1, size, size)
    subsequent = torch.triu(torch.ones(attn_shape, dtype=torch.bool), diagonal=1)
    return subsequent

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = self.d_v = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.attn = ScaledDotProductAttention(dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B, Lq, d = q.shape
        Lk = k.size(1)
        h = self.n_heads
        Q = self.w_q(q).view(B, Lq, h, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, Lk, h, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, Lk, h, self.d_v).transpose(1, 2)
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
        out = self.attn(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, Lq, h * self.d_v)
        out = self.w_o(out)
        return self.dropout(out)

class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.lin1 = nn.Linear(d_model, d_ff)
        self.lin2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.lin2(self.dropout(F.gelu(self.lin1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, src_mask=None):
        attn_out = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.drop(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.drop(ffn_out))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, memory, tgt_mask=None, memory_mask=None):
        x2 = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.drop(x2))
        x2 = self.cross_attn(x, memory, memory, mask=memory_mask)
        x = self.norm2(x + self.drop(x2))
        x2 = self.ffn(x)
        x = self.norm3(x + self.drop(x2))
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=8192):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:, :L, :]

class AIAINYTransformerEncDecBinary(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_enc_layers=2, n_dec_layers=2, d_ff=128, dropout=0.1):
        super().__init__()
        self.src_proj = nn.Linear(1, d_model)
        self.pe_src = PositionalEncoding(d_model)
        self.pe_tgt = PositionalEncoding(d_model)
        self.bos = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.bos, mean=0.0, std=0.02)

        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)
        ])
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)
        ])
        self.head = nn.Linear(d_model, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):  # src: [B, L, 1]
        B, L, _ = src.shape
        src = self.src_proj(src)
        src = self.pe_src(src)
        enc = src
        for layer in self.enc_layers:
            enc = layer(enc, src_mask=None)

        tgt = self.bos.expand(B, 1, -1)
        tgt = self.pe_tgt(tgt)
        tgt_mask = subsequent_mask(tgt.size(1)).to(tgt.device)
        mem_mask = None
        dec = tgt
        for layer in self.dec_layers:
            dec = layer(dec, enc, tgt_mask=tgt_mask, memory_mask=mem_mask)

        logit = self.head(dec[:, 0, :])
        return logit.squeeze(-1)


## 5) Train & Evaluation (+ best-F1 threshold on validation)

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for X, Y in loader:
        X = X.to(device)
        Y = Y.to(device).float()
        logit = model(X)
        loss = criterion(logit, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * X.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def infer_probs(model, loader, device):
    model.eval()
    probs, labels = [], []
    for X, Y in loader:
        X = X.to(device)
        p = torch.sigmoid(model(X)).cpu().numpy()
        probs.append(p)
        labels.append(Y.numpy())
    return np.concatenate(probs), np.concatenate(labels)

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
def metrics_prfa(y_true: np.ndarray, y_prob: np.ndarray, thr: float = 0.5):
    y_pred = (y_prob >= thr).astype(np.int64)
    P = precision_score(y_true, y_pred, zero_division=0)
    R = recall_score(y_true, y_pred, zero_division=0)
    F1 = f1_score(y_true, y_pred, zero_division=0)
    # AUROC는 예측 확률값(y_prob)을 직접 사용
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:  # 단일 클래스만 있는 경우
        auroc = 0.0
    return {
        "Precision": float(P),
        "Recall": float(R),
        "F1": float(F1),
        "AUROC": float(auroc)
    }
def find_best_threshold(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for thr in grid:
        f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr), float(best_f1)


## 6) Per-file Training & Testing

In [ ]:
def run_one_file(csv_path: str, cfg):
    x, y = load_wsd(csv_path)
    n = len(x)
    if n < cfg.seq_len + 10:
        raise ValueError(f"Series too short for seq_len={cfg.seq_len}: {csv_path}")

    n_tr, n_va = int(0.6 * n), int(0.8 * n)
    x_tr, x_va, x_te = x[:n_tr], x[n_tr:n_va], x[n_va:]
    y_tr, y_va, y_te = y[:n_tr], y[n_tr:n_va], y[n_va:]

    rs = fit_robust_scaler(x_tr, cfg)
    x_tr = apply_scaler(x_tr, rs)
    x_va = apply_scaler(x_va, rs)
    x_te = apply_scaler(x_te, rs)

    L = cfg.seq_len
    Xtr, Ytr, _ = make_windows(x_tr, y_tr, L)
    Xva, Yva, _ = make_windows(np.concatenate([x_tr, x_va]),
                               np.concatenate([y_tr, y_va]), L)
    Xte, Yte, _ = make_windows(np.concatenate([x_tr, x_va, x_te]),
                               np.concatenate([y_tr, y_va, y_te]), L)

    ds_tr, ds_va, ds_te = WindowDataset(Xtr, Ytr), WindowDataset(Xva, Yva), WindowDataset(Xte, Yte)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=cfg.batch, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=cfg.batch, shuffle=False)

    device = cfg.device if (cfg.device == "cpu" or torch.cuda.is_available()) else "cpu"
    model = AIAINYTransformerEncDecBinary(
        d_model=cfg.d_model, n_heads=cfg.n_heads,
        n_enc_layers=cfg.n_enc_layers, n_dec_layers=cfg.n_dec_layers,
        d_ff=cfg.dim_ff, dropout=cfg.dropout
    ).to(device)
    pos_w = compute_pos_weight(Ytr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, cfg.epochs + 1):
        _ = train_one_epoch(model, dl_tr, optimizer, criterion, device)
        p_va, y_va = infer_probs(model, dl_va, device)
        f1 = metrics_prfa(y_va, p_va)["F1"]
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if ep % 5 == 0 or bad == 0:
            print(f"[{os.path.basename(csv_path)}] epoch {ep} valF1={f1:.4f} best={best_f1:.4f}, bad={bad}")
        if bad >= cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_va, y_va = infer_probs(model, dl_va, device)
    best_thr, _ = find_best_threshold(y_va, p_va)

    p_te, y_te = infer_probs(model, dl_te, device)
    return metrics_prfa(y_te, p_te, thr=best_thr)


## 7) Evaluate over all A1 files

In [21]:
def evaluate_all(cfg):
    paths = sorted(glob.glob(os.path.join(cfg.a1_dir, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No files matched: {cfg.a1_dir}/*.csv")
    rows = []
    for p in paths:
        print(f"\n### Processing: {p}")
        m = run_one_file(p, cfg)
        rows.append(m)
        print("=> Test metrics:", json.dumps(m, ensure_ascii=False))
    keys = rows[0].keys()
    macro = {k: float(np.mean([r[k] for r in rows])) for k in keys}
    return paths, rows, macro

paths, rows, macro = evaluate_all(cfg)

print("\n# Per-file metrics (Precision/Recall/F1/Accuracy)")
for p, m in zip(paths, rows):
    print(p, json.dumps(m, ensure_ascii=False))

print("\n# Macro mean (simple average over files)")
print(json.dumps(macro, ensure_ascii=False))



### Processing: ./TSB-AD-U/WSD/029_WSD_id_1_WebService_tr_4559_1st_10201.csv


ValueError: Required columns not found in CSV. Columns: ['Data', 'Label']